# Gravitational Lens Substructure Classifier
**Before running:** `Runtime -> Change runtime type -> T4 GPU`

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile

ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
print('Zip found:', os.path.exists(ZIP_PATH))
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
print('Extracted.')

for split in ['train', 'val']:
    for cls in ['no', 'sphere', 'vort']:
        p = f'/content/dataset/{split}/{cls}'
        n = len(os.listdir(p)) if os.path.exists(p) else 'MISSING'
        print(f'  dataset/{split}/{cls}: {n}')

In [ ]:
!pip install -q tqdm scikit-learn

In [ ]:
import os, time, json, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
import torchvision.transforms.functional as TF
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from tqdm.notebook import tqdm

CLASS_NAMES   = ['no', 'sphere', 'vort']
CLASS_TO_IDX  = {n: i for i, n in enumerate(CLASS_NAMES)}
IMAGENET_MEAN = 0.449
IMAGENET_STD  = 0.226


class LensDataset(Dataset):
    def __init__(self, root, augment=False):
        self.augment = augment
        self.samples = []
        for cls in CLASS_NAMES:
            d = os.path.join(root, cls)
            for f in sorted(os.listdir(d)):
                if f.endswith('.npy'):
                    self.samples.append((os.path.join(d, f), CLASS_TO_IDX[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        t = torch.from_numpy(np.load(path).astype(np.float32))
        if self.augment:
            if random.random() > 0.5: t = TF.hflip(t)
            if random.random() > 0.5: t = TF.vflip(t)
            k = random.randint(0, 3)
            if k: t = torch.rot90(t, k, dims=[1, 2])
            t = TF.rotate(t, random.uniform(-15, 15))
            t = torch.clamp(t + random.uniform(-0.05, 0.05), 0.0, 1.0)
        t = (t - IMAGENET_MEAN) / IMAGENET_STD
        return t, label


class LensEfficientNet(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=False, dropout=0.4):
        super().__init__()
        bb = models.efficientnet_b0(
            weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        old = bb.features[0][0]
        new = nn.Conv2d(1, old.out_channels, old.kernel_size,
                        old.stride, old.padding, bias=False)
        if pretrained:
            new.weight.data = old.weight.data.mean(dim=1, keepdim=True)
        bb.features[0][0] = new
        in_f = bb.classifier[1].in_features
        bb.classifier = nn.Identity()
        self.backbone = bb
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_f, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(512, 3),
        )
        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

    def forward(self, x):
        return self.head(self.backbone(x))


print('Classes defined OK')

In [ ]:
DATA_ROOT     = '/content/dataset'
EPOCHS        = 75
WARMUP_EPOCHS = 3
BATCH_SIZE    = 256
LR            = 1e-3
BACKBONE_LR   = 1e-4
WEIGHT_DECAY  = 1e-4
DROPOUT       = 0.4
WORKERS       = 2
SAVE_DIR      = 'checkpoints'
DRIVE_OUT     = '/content/drive/MyDrive/lens_results'

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

train_ds = LensDataset(os.path.join(DATA_ROOT, 'train'), augment=True)
val_ds   = LensDataset(os.path.join(DATA_ROOT, 'val'),   augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=WORKERS, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=WORKERS, pin_memory=True, persistent_workers=True)
print(f'train={len(train_ds):,}  val={len(val_ds):,}  steps/epoch={len(train_loader)}')

model = LensEfficientNet(pretrained=True, freeze_backbone=True, dropout=DROPOUT).to(device)
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Total    : {sum(p.numel() for p in model.parameters()):,}')

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler('cuda')
steps  = len(train_loader)

def make_scheduler(opt, n_epochs, max_lr):
    return torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=max_lr, steps_per_epoch=steps,
        epochs=n_epochs, pct_start=0.3, anneal_strategy='cos')

scheduler = make_scheduler(optimizer, max(WARMUP_EPOCHS, 1), LR)

def compute_auc(labels, probs):
    y = label_binarize(labels, classes=[0, 1, 2])
    return roc_auc_score(y, probs, average='macro', multi_class='ovr')


def train_epoch(epoch):
    model.train()
    total_loss, total = 0.0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} train', leave=False)
    for imgs, lbls in pbar:
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            loss = criterion(model(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * lbls.size(0)
        total      += lbls.size(0)
        pbar.set_postfix(loss=f'{total_loss/total:.4f}', lr=f'{scheduler.get_last_lr()[0]:.1e}')
    return total_loss / total


@torch.no_grad()
def val_epoch(epoch):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    pbar = tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS}   val', leave=False)
    for imgs, lbls in pbar:
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        with autocast('cuda'):
            logits = model(imgs)
            loss   = criterion(logits, lbls)
        probs = torch.softmax(logits.float(), 1)
        total_loss += loss.item() * lbls.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += lbls.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(lbls.cpu().numpy())
        pbar.set_postfix(loss=f'{total_loss/total:.4f}', acc=f'{correct/total:.4f}')
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    return total_loss/total, correct/total, compute_auc(all_labels, all_probs), all_probs, all_labels


print('Setup complete. Ready to train.')

In [ ]:
history   = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_auc  = 0.0
best_ckpt = f'{SAVE_DIR}/best_model.pt'
fine_tune_epochs = EPOCHS - WARMUP_EPOCHS

print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Val Loss":>8}  {"Val Acc":>7}  {"Val AUC":>7}  {"Time":>6}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    if epoch == WARMUP_EPOCHS + 1:
        print(f'\n[phase] Unfreezing backbone at epoch {epoch}')
        model.unfreeze_backbone()
        optimizer = torch.optim.AdamW(
            [{'params': model.backbone.parameters(), 'lr': BACKBONE_LR},
             {'params': model.head.parameters(),     'lr': LR}],
            weight_decay=WEIGHT_DECAY)
        scheduler = make_scheduler(optimizer, fine_tune_epochs, [BACKBONE_LR, LR])
        scaler = GradScaler('cuda')

    train_loss = train_epoch(epoch)
    val_loss, val_acc, val_auc, val_probs, val_labels = val_epoch(epoch)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    tag = '  << best' if val_auc > best_auc else ''
    print(f'{epoch:>6}  {train_loss:>10.4f}  {val_loss:>8.4f}  {val_acc:>7.4f}  {val_auc:>7.4f}  {time.time()-t0:>5.1f}s{tag}')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_auc': val_auc, 'val_acc': val_acc}, best_ckpt)

print(f'\nBest Val AUC = {best_auc:.4f}')
with open(f'{SAVE_DIR}/history.json', 'w') as f:
    json.dump(history, f)
np.savez(f'{SAVE_DIR}/val_preds.npz', probs=val_probs, labels=val_labels)
print('Training complete.')

In [ ]:
# ── Save to Drive immediately after training ──────────────────────────
import shutil
for f in ['best_model.pt', 'history.json', 'val_preds.npz']:
    shutil.copy(f'{SAVE_DIR}/{f}', DRIVE_OUT)
print(f'Saved to Drive -> {DRIVE_OUT}')

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

data   = np.load(f'{SAVE_DIR}/val_preds.npz')
probs  = data['probs']
labels = data['labels']
preds  = probs.argmax(1)
y_bin  = label_binarize(labels, classes=[0, 1, 2])
COLORS = ['#e41a1c', '#377eb8', '#4daf4a']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ROC Curves - Gravitational Lens Classifier', fontsize=13)

all_fpr  = np.linspace(0, 1, 300)
mean_tpr = np.zeros(300)

for i, name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
    score = auc(fpr, tpr)
    mean_tpr += np.interp(all_fpr, fpr, tpr)
    for ax in axes:
        ax.plot(fpr, tpr, color=COLORS[i], lw=2, label=f'{name}  (AUC={score:.4f})')

mean_tpr /= 3
macro_auc = auc(all_fpr, mean_tpr)
for ax in axes:
    ax.plot(all_fpr, mean_tpr, 'navy', lw=2.5, ls='--', label=f'Macro avg (AUC={macro_auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k:', lw=1.2, label='Random (0.50)')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_title('Full ROC')
axes[1].set_xlim([-0.005, 0.3])
axes[1].set_ylim([0.7, 1.005])
axes[1].set_title('Zoomed (top-left)')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Macro AUC : {macro_auc:.4f}')
print(f'Accuracy  : {(preds == labels).mean():.4f}')
for i, name in enumerate(CLASS_NAMES):
    print(f'AUC [{name:>6}]: {roc_auc_score(y_bin[:, i], probs[:, i]):.4f}')

In [ ]:
# ── Confusion matrix + training history ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(labels, preds)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix')

with open(f'{SAVE_DIR}/history.json') as f:
    h = json.load(f)
ep = range(1, len(h['val_auc']) + 1)

axes[1].plot(ep, h['train_loss'], label='Train', color='#e41a1c')
axes[1].plot(ep, h['val_loss'],   label='Val',   color='#377eb8')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(ep, h['val_auc'], label='Val AUC', color='#4daf4a', lw=2)
axes[2].plot(ep, h['val_acc'], label='Val Acc', color='#ff7f00', lw=2)
axes[2].set_title('Val AUC & Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save plots to Drive ───────────────────────────────────────────────
import shutil
for f in ['roc_curves.png', 'training_summary.png']:
    shutil.copy(f, DRIVE_OUT)
print(f'Plots saved to Drive -> {DRIVE_OUT}')